# Interseção SICOR Cerrado × Áreas Quilombolas por ID_GLEBA2

Este notebook cruza o arquivo `sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv` com o shapefile `Áreas de Quilombolas.shp`.

## Regra principal

- cada `ID_GLEBA2` distinto é submetido à interseção espacial somente uma vez;
- quando o mesmo `ID_GLEBA2` aparece em várias linhas, o resultado calculado é reaproveitado;
- antes do processamento, o notebook valida se um mesmo `ID_GLEBA2` está sempre associado à mesma geometria;
- a deduplicação técnica usa a chave `ID_GLEBA2 + QUIL_ID_INTERNO`, e não `REF_BACEN`.

## Saídas

1. **Resultado espacial único:** uma linha por `ID_GLEBA2 × área quilombola`.
2. **Resultado detalhado expandido:** associa a resposta espacial a todas as linhas originais que possuem aquele `ID_GLEBA2`.
3. **Resumo por ID_GLEBA2:** contém todos os IDs, inclusive os que não intersectaram áreas quilombolas.

Os atributos do shapefile são preservados com o prefixo `QUIL_`. Assim, uma geometria repetida não é testada novamente, mas nenhum vínculo com `REF_BACEN`, ordem, índice ou ano é perdido.


In [1]:
# ============================================================
# 1. INSTALAR, IMPORTAR E MONTAR O GOOGLE DRIVE
# ============================================================

%pip install -q --upgrade geopandas pyogrio shapely

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import force_2d, intersection as shapely_intersection, make_valid, set_precision
from shapely.errors import GEOSException

warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 180)

print('Ambiente preparado.')


Mounted at /content/drive
Ambiente preparado.


In [2]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

ARQUIVO_SICOR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/"
    "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
)

ARQUIVO_QUILOMBOLA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Base/Áreas de Quilombolas.shp"
)

# Pasta nova para não sobrescrever resultados anteriores.
PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/intersecoes/quilombolas/"
    "resultados_por_id_gleba2"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_INTERSECAO_POR_ID = (
    PASTA_SAIDA / "intersecao_quilombolas_unica_por_id_gleba2.csv"
)

ARQUIVO_INTERSECAO_EXPANDIDA = (
    PASTA_SAIDA / "intersecao_sicor_quilombolas_expandida_por_id_gleba2.csv"
)

ARQUIVO_RESUMO_POR_ID = (
    PASTA_SAIDA / "resumo_quilombolas_por_id_gleba2.csv"
)

CRS_SICOR = "EPSG:4326"
CRS_AREA = "EPSG:5880"
TAMANHO_BLOCO = 20_000
SEPARADOR_SAIDA = ";"

for caminho in [ARQUIVO_SICOR, ARQUIVO_QUILOMBOLA]:
    print(f"{'OK' if caminho.exists() else 'NÃO ENCONTRADO'} | {caminho}")

if not ARQUIVO_SICOR.exists():
    raise FileNotFoundError(f"CSV do SICOR não encontrado: {ARQUIVO_SICOR}")

if not ARQUIVO_QUILOMBOLA.exists():
    raise FileNotFoundError(
        f"Shapefile de áreas quilombolas não encontrado: "
        f"{ARQUIVO_QUILOMBOLA}"
    )

for extensao in [".dbf", ".shx", ".prj"]:
    componente = ARQUIVO_QUILOMBOLA.with_suffix(extensao)
    print(f"{'OK' if componente.exists() else 'ATENÇÃO'} | {componente.name}")

print("\nPasta de saída:")
print(PASTA_SAIDA)


OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/Áreas de Quilombolas.shp
OK | Áreas de Quilombolas.dbf
OK | Áreas de Quilombolas.shx
OK | Áreas de Quilombolas.prj

Pasta de saída:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/quilombolas/resultados_por_id_gleba2


In [3]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

CODIFICACOES = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
SEPARADORES = [';', ',', '\t', '|']


def remover_acentos(texto):
    return ''.join(
        caractere
        for caractere in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip().replace('#', '')
    nome = re.sub(r'[^A-Z0-9]+', '_', nome)
    return re.sub(r'_+', '_', nome).strip('_')


def normalizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f'A normalização produziu colunas duplicadas: {duplicadas}')

    return df


def detectar_csv(caminho):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    dtype=str,
                    low_memory=False,
                )

                if teste.shape[1] <= 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        'Não foi possível detectar a estrutura do CSV. '
        f'Último erro: {ultimo_erro}'
    )


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    for candidato in [normalizar_nome_coluna(c) for c in candidatos]:
        if candidato in colunas:
            return candidato

    if obrigatoria:
        raise KeyError(
            'Nenhuma das colunas esperadas foi encontrada: '
            f'{[normalizar_nome_coluna(c) for c in candidatos]}'
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lstrip('\ufeff')
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, '<NA>': pd.NA})
    )


def corrigir_geometrias(gdf):
    quantidade_entrada = len(gdf)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    mascara_invalida = ~gdf.geometry.is_valid
    quantidade_invalidas = int(mascara_invalida.sum())

    if quantidade_invalidas:
        try:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].make_valid()
            )
        except Exception:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].buffer(0)
            )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    quantidade_descartadas = quantidade_entrada - len(gdf)
    return gdf, quantidade_invalidas, quantidade_descartadas


def serie_wkt_para_geometria(serie):
    return gpd.GeoSeries.from_wkt(
        serie.astype('string'),
        crs=CRS_SICOR,
        on_invalid='ignore',
    )


def preparar_geometria_topologica(geometria, precisao=None):
    if geometria is None:
        return None

    try:
        if geometria.is_empty:
            return None
        geometria = force_2d(geometria)

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        if precisao is not None:
            geometria = set_precision(
                geometria,
                grid_size=precisao,
                mode='valid_output',
            )

        if geometria is None or geometria.is_empty:
            return None

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        return geometria

    except Exception:
        try:
            geometria = geometria.buffer(0)
            return None if geometria.is_empty else geometria
        except Exception:
            return None


def calcular_intersecoes_seguras(geometrias_esquerda, geometrias_direita, crs):
    resultados = []
    quantidade_recuperadas = 0
    quantidade_falhas = 0

    for geometria_esquerda, geometria_direita in zip(
        geometrias_esquerda.to_numpy(),
        geometrias_direita.to_numpy(),
    ):
        esquerda = preparar_geometria_topologica(geometria_esquerda, 0.01)
        direita = preparar_geometria_topologica(geometria_direita, 0.01)

        if esquerda is None or direita is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        intersecao = None
        recuperada = False

        try:
            intersecao = shapely_intersection(esquerda, direita, grid_size=0.01)
        except (GEOSException, ValueError, TypeError):
            recuperada = True

            try:
                esquerda_10 = preparar_geometria_topologica(esquerda, 0.10)
                direita_10 = preparar_geometria_topologica(direita, 0.10)

                if esquerda_10 is not None and direita_10 is not None:
                    intersecao = shapely_intersection(
                        esquerda_10,
                        direita_10,
                        grid_size=0.10,
                    )
            except (GEOSException, ValueError, TypeError):
                intersecao = None

            if intersecao is None:
                try:
                    intersecao = esquerda.buffer(0).intersection(direita.buffer(0))
                except Exception:
                    intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        try:
            if not intersecao.is_empty and not intersecao.is_valid:
                intersecao = make_valid(intersecao)
        except Exception:
            try:
                intersecao = intersecao.buffer(0)
            except Exception:
                intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
        else:
            resultados.append(intersecao)
            if recuperada:
                quantidade_recuperadas += 1

    return (
        gpd.GeoSeries(resultados, index=pd.RangeIndex(len(resultados)), crs=crs),
        quantidade_recuperadas,
        quantidade_falhas,
    )


In [4]:
# ============================================================
# 4. LER E PREPARAR AS ÁREAS QUILOMBOLAS
# ============================================================

quilombolas = gpd.read_file(ARQUIVO_QUILOMBOLA, engine='pyogrio')
crs_original_quil = quilombolas.crs
coluna_geometria_original = quilombolas.geometry.name

if crs_original_quil is None:
    raise ValueError(
        'O shapefile Áreas de Quilombolas.shp não possui CRS. Verifique se o arquivo .prj '
        'está na mesma pasta do shapefile.'
    )

quilombolas = normalizar_colunas(quilombolas)
coluna_geometria_normalizada = normalizar_nome_coluna(coluna_geometria_original)

colunas_geometricas = [
    coluna
    for coluna in quilombolas.columns
    if isinstance(quilombolas[coluna].dtype, gpd.array.GeometryDtype)
]

if coluna_geometria_normalizada not in quilombolas.columns:
    if not colunas_geometricas:
        raise KeyError('Nenhuma coluna geométrica foi localizada no shapefile de áreas quilombolas.')
    coluna_geometria_normalizada = colunas_geometricas[0]

quilombolas = gpd.GeoDataFrame(
    quilombolas,
    geometry=coluna_geometria_normalizada,
    crs=crs_original_quil,
)

if quilombolas.geometry.name != 'geometry':
    quilombolas = quilombolas.rename_geometry('geometry')

quantidade_quil_original = len(quilombolas)
quilombolas, invalidas_quil, descartadas_quil = corrigir_geometrias(quilombolas)
quilombolas = quilombolas.reset_index(drop=True)
quilombolas['QUIL_ID_INTERNO'] = np.arange(1, len(quilombolas) + 1)

renomear_quil = {
    coluna: f'QUIL_{coluna}'
    for coluna in quilombolas.columns
    if coluna not in {'geometry', 'QUIL_ID_INTERNO'}
}
quilombolas = quilombolas.rename(columns=renomear_quil).set_geometry('geometry')
quilombolas = quilombolas.to_crs(CRS_SICOR).reset_index(drop=True)

quilombolas_area = quilombolas.to_crs(CRS_AREA)
quilombolas['QUIL_AREA_TOTAL_HA'] = quilombolas_area.geometry.area / 10_000

COLUNAS_QUIL_SAIDA = [coluna for coluna in quilombolas.columns if coluna != 'geometry']

print('Feições originais de áreas quilombolas:', f'{quantidade_quil_original:,}')
print('Geometrias inválidas corrigidas:', f'{invalidas_quil:,}')
print('Geometrias descartadas:', f'{descartadas_quil:,}')
print('Feições válidas finais:', f'{len(quilombolas):,}')
print('CRS espacial:', quilombolas.crs)
print('CRS para áreas:', CRS_AREA)

display(quilombolas.head())


Feições originais de áreas quilombolas: 439
Geometrias inválidas corrigidas: 1
Geometrias descartadas: 0
Feições válidas finais: 439
CRS espacial: EPSG:4326
CRS para áreas: EPSG:5880


,QUIL_CD_QUILOMB,QUIL_CD_SR,QUIL_NR_PROCESS,QUIL_NM_COMUNID,QUIL_NM_MUNICIP,QUIL_CD_UF,QUIL_DT_PUBLICA,QUIL_DT_PUBLIC1,QUIL_NR_FAMILIA,QUIL_DT_TITULAC,QUIL_NR_AREA_HA,QUIL_NR_PERIMET,QUIL_CD_SIPRA,QUIL_OB_DESCRIC,QUIL_ST_TITULAD,QUIL_DT_DECRETO,QUIL_TP_LEVANTA,QUIL_NR_ESCALAO,QUIL_AREA_CALC,QUIL_PERIMETRO,QUIL_ESFERA,QUIL_FASE,QUIL_RESPONSAVE,geometry,QUIL_ID_INTERNO,QUIL_AREA_TOTAL_HA
0,NaN,01,None,TQ Sitio São João - parte A,Barcarena,PA,None,None,NaN,None,NaN,NaN,None,None,None,None,None,None,NaN,NaN,None,None,None,"POLYGON ((-48.70403 -1.52458, -48.70387 -1.52549, -48.70355 -1.52723, -48.70442 -1.52737, -48.70431 -1.52797, -48.7054 -1.52817, -48.70515 -1.52899, -48.70511 -1.52916, -48.704...",1,30.765338
1,NaN,01,None,TQ Sitio São João parte B,Barcarena,PA,None,None,NaN,None,NaN,NaN,None,None,None,None,None,None,NaN,NaN,None,None,None,"POLYGON ((-48.7032 -1.52631, -48.70318 -1.52642, -48.70344 -1.52647, -48.70347 -1.52636, -48.7032 -1.52631))",2,0.038697
2,NaN,06,None,TQ Nogueira,Montes Claros,MG,None,None,NaN,None,12.7146,NaN,None,None,None,None,None,None,NaN,NaN,None,None,None,"POLYGON ((-43.82701 -16.73528, -43.82721 -16.73528, -43.82727 -16.73557, -43.82731 -16.73572, -43.82733 -16.73602, -43.82723 -16.73602, -43.82723 -16.73606, -43.8272 -16.7363, ...",3,12.899475
3,NaN,01,None,TQ Barcarena,Barcarena,PA,None,None,NaN,None,29.7885,NaN,None,None,None,None,None,None,NaN,NaN,None,None,None,"POLYGON ((-48.68961 -1.51998, -48.68839 -1.51965, -48.68647 -1.52398, -48.68277 -1.5234, -48.68229 -1.52583, -48.68836 -1.527, -48.68961 -1.51998))",4,29.917422
4,NaN,12,None,TQ_ALDEIA_VELHA,Pirapemas,MA,None,None,NaN,None,7772.7144,NaN,None,None,None,None,None,None,NaN,NaN,None,None,None,"POLYGON ((-44.2155 -3.84495, -44.21561 -3.84485, -44.21147 -3.84053, -44.21677 -3.83347, -44.21851 -3.83108, -44.22007 -3.82901, -44.22237 -3.82588, -44.22395 -3.8237, -44.2254...",5,7886.811178


In [5]:
# ============================================================
# 5. DETECTAR E VALIDAR AS COLUNAS DO SICOR
# ============================================================

encoding_sicor, separador_sicor, colunas_sicor = detectar_csv(ARQUIVO_SICOR)

COLUNA_ID_GLEBA2 = localizar_coluna(
    colunas_sicor,
    ['ID_GLEBA2', 'ID_GLEBA_2'],
)

COLUNA_GEOMETRIA_SICOR = localizar_coluna(
    colunas_sicor,
    ['GT_GEOMETRIA', 'GT_GEOMETRIA_REPRESENTATIVA', 'GEOMETRIA', 'GEOMETRY', 'WKT'],
)

COLUNA_REF_BACEN = localizar_coluna(
    colunas_sicor,
    ['REF_BACEN', '#REF_BACEN', 'CD_REF_BACEN'],
    obrigatoria=False,
)

print('Codificação:', encoding_sicor)
print('Separador:', repr(separador_sicor))
print('Coluna-chave:', COLUNA_ID_GLEBA2)
print('Coluna de geometria:', COLUNA_GEOMETRIA_SICOR)
print('Coluna REF_BACEN:', COLUNA_REF_BACEN)
print('Quantidade de colunas:', len(colunas_sicor))

if COLUNA_ID_GLEBA2 == COLUNA_GEOMETRIA_SICOR:
    raise RuntimeError('A coluna do ID e a coluna geométrica não podem ser a mesma.')


Codificação: utf-8-sig
Separador: ';'
Coluna-chave: ID_GLEBA2
Coluna de geometria: GT_GEOMETRIA
Coluna REF_BACEN: REF_BACEN
Quantidade de colunas: 27


In [6]:
# ============================================================
# 6. INTERSECTAR UMA ÚNICA VEZ CADA ID_GLEBA2
# ============================================================

# Guarda somente uma assinatura numérica da geometria de cada ID.
# Isso permite reconhecer IDs já processados sem manter todos os WKT na memória.
assinatura_geometria_por_id = {}
resultados_intersecao = []

total_linhas_lidas = 0
total_linhas_com_id_e_geometria = 0
total_ids_novos = 0
total_repeticoes_puladas = 0
total_ids_geometria_valida = 0
total_geometrias_invalidas_corrigidas = 0
total_geometrias_descartadas = 0
total_candidatos_espaciais = 0
total_intersecoes_validas = 0
total_intersecoes_recuperadas = 0
total_falhas_topologicas = 0
total_contatos_sem_area = 0

leitor = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    bloco = normalizar_colunas(bloco)
    total_linhas_lidas += len(bloco)

    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    bloco[COLUNA_GEOMETRIA_SICOR] = bloco[COLUNA_GEOMETRIA_SICOR].astype('string').str.strip()

    bloco = bloco.dropna(
        subset=[COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
    ).copy()

    total_linhas_com_id_e_geometria += len(bloco)

    if bloco.empty:
        print(f'Bloco {numero_bloco:03d}: sem ID_GLEBA2 e geometria válidos')
        continue

    bloco['_HASH_GEOMETRIA'] = (
        pd.util.hash_pandas_object(
            bloco[COLUNA_GEOMETRIA_SICOR],
            index=False,
        )
        .astype('uint64')
    )

    # Validação dentro do próprio bloco.
    geometrias_por_id_no_bloco = bloco.groupby(
        COLUNA_ID_GLEBA2,
        dropna=False,
    )['_HASH_GEOMETRIA'].nunique()

    ids_inconsistentes = geometrias_por_id_no_bloco[
        geometrias_por_id_no_bloco > 1
    ].index.tolist()

    if ids_inconsistentes:
        raise ValueError(
            'Um mesmo ID_GLEBA2 foi associado a geometrias diferentes. '
            f'Primeiros IDs inconsistentes: {ids_inconsistentes[:10]}'
        )

    representantes = bloco.drop_duplicates(
        subset=[COLUNA_ID_GLEBA2],
        keep='first',
    ).reset_index(drop=True)

    mascara_id_novo = []

    # Validação também contra IDs que apareceram em blocos anteriores.
    for id_atual, hash_atual in representantes[
        [COLUNA_ID_GLEBA2, '_HASH_GEOMETRIA']
    ].itertuples(index=False, name=None):
        hash_atual = int(hash_atual)

        if id_atual in assinatura_geometria_por_id:
            if assinatura_geometria_por_id[id_atual] != hash_atual:
                raise ValueError(
                    f'O ID_GLEBA2 {id_atual} apareceu com geometrias diferentes '
                    'em blocos distintos do CSV.'
                )
            mascara_id_novo.append(False)
        else:
            assinatura_geometria_por_id[id_atual] = hash_atual
            mascara_id_novo.append(True)

    representantes_novos = representantes.loc[
        mascara_id_novo
    ].copy().reset_index(drop=True)

    total_ids_novos += len(representantes_novos)
    total_repeticoes_puladas += len(bloco) - len(representantes_novos)

    if representantes_novos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            'todos os IDs já haviam sido processados'
        )
        del bloco, representantes, representantes_novos
        gc.collect()
        continue

    geometria = serie_wkt_para_geometria(
        representantes_novos[COLUNA_GEOMETRIA_SICOR]
    )

    gdf_ids = gpd.GeoDataFrame(
        representantes_novos[
            [COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
        ].copy(),
        geometry=geometria,
        crs=CRS_SICOR,
    )

    gdf_ids, invalidas_corrigidas, descartadas = corrigir_geometrias(gdf_ids)
    gdf_ids = gdf_ids.reset_index(drop=True)

    total_geometrias_invalidas_corrigidas += invalidas_corrigidas
    total_geometrias_descartadas += descartadas
    total_ids_geometria_valida += len(gdf_ids)

    if gdf_ids.empty:
        print(
            f'Bloco {numero_bloco:03d}: IDs novos={len(representantes_novos):,} | '
            'nenhuma geometria válida'
        )
        del bloco, representantes, representantes_novos, gdf_ids
        gc.collect()
        continue

    candidatos = gpd.sjoin(
        gdf_ids,
        quilombolas,
        how='inner',
        predicate='intersects',
    ).reset_index(drop=True)

    total_candidatos_espaciais += len(candidatos)

    if candidatos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            f'IDs novos={len(representantes_novos):,} | interseções=0 | '
            f'repetições puladas={len(bloco) - len(representantes_novos):,}'
        )
        del bloco, representantes, representantes_novos, gdf_ids, candidatos
        gc.collect()
        continue

    candidatos['GEOMETRIA_QUILOMBOLA'] = (
        candidatos['index_right'].map(quilombolas.geometry)
    )

    candidatos = candidatos.dropna(
        subset=['GEOMETRIA_QUILOMBOLA']
    ).reset_index(drop=True)

    geometria_gleba_wgs84 = gpd.GeoSeries(
        candidatos.geometry.to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_quilombola_wgs84 = gpd.GeoSeries(
        candidatos['GEOMETRIA_QUILOMBOLA'].to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_gleba_area = geometria_gleba_wgs84.to_crs(CRS_AREA)
    geometria_quilombola_area = geometria_quilombola_wgs84.to_crs(CRS_AREA)

    geometria_gleba_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_gleba_area],
        index=pd.RangeIndex(len(geometria_gleba_area)),
        crs=CRS_AREA,
    )

    geometria_quilombola_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_quilombola_area],
        index=pd.RangeIndex(len(geometria_quilombola_area)),
        crs=CRS_AREA,
    )

    (
        geometria_intersecao_area,
        recuperadas_bloco,
        falhas_bloco,
    ) = calcular_intersecoes_seguras(
        geometria_gleba_area,
        geometria_quilombola_area,
        CRS_AREA,
    )

    total_intersecoes_recuperadas += recuperadas_bloco
    total_falhas_topologicas += falhas_bloco

    candidatos['AREA_GLEBA_HA_CALCULADA'] = (
        geometria_gleba_area.area.to_numpy() / 10_000
    )
    candidatos['AREA_INTERSECAO_HA'] = (
        geometria_intersecao_area.area.to_numpy() / 10_000
    )
    candidatos['PERCENTUAL_GLEBA_INTERSECTADA'] = np.where(
        candidatos['AREA_GLEBA_HA_CALCULADA'] > 0,
        candidatos['AREA_INTERSECAO_HA']
        / candidatos['AREA_GLEBA_HA_CALCULADA']
        * 100,
        np.nan,
    )

    mascara_area_positiva = (
        candidatos['AREA_INTERSECAO_HA'].fillna(0).gt(0).to_numpy()
    )
    total_contatos_sem_area += int((~mascara_area_positiva).sum())

    if not mascara_area_positiva.any():
        print(
            f'Bloco {numero_bloco:03d}: candidatos={len(candidatos):,} | '
            'nenhuma interseção com área positiva'
        )
        del (
            bloco, representantes, representantes_novos, gdf_ids, candidatos,
            geometria_gleba_wgs84, geometria_quilombola_wgs84,
            geometria_gleba_area, geometria_quilombola_area,
            geometria_intersecao_area, mascara_area_positiva,
        )
        gc.collect()
        continue

    candidatos = candidatos.loc[mascara_area_positiva].reset_index(drop=True)

    geometria_intersecao_area_valida = gpd.GeoSeries(
        geometria_intersecao_area.to_numpy()[mascara_area_positiva],
        index=pd.RangeIndex(int(mascara_area_positiva.sum())),
        crs=CRS_AREA,
    )
    geometria_intersecao_wgs84 = geometria_intersecao_area_valida.to_crs(CRS_SICOR)

    candidatos['GEOMETRIA_QUILOMBOLA_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in candidatos['GEOMETRIA_QUILOMBOLA']
    ]
    candidatos['GEOMETRIA_INTERSECAO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in geometria_intersecao_wgs84
    ]
    candidatos['TIPO_IMPEDIMENTO'] = 'Território Quilombola'

    colunas_resultado = [
        COLUNA_ID_GLEBA2,
        *COLUNAS_QUIL_SAIDA,
        'AREA_GLEBA_HA_CALCULADA',
        'AREA_INTERSECAO_HA',
        'PERCENTUAL_GLEBA_INTERSECTADA',
        'GEOMETRIA_QUILOMBOLA_WKT',
        'GEOMETRIA_INTERSECAO_WKT',
        'TIPO_IMPEDIMENTO',
    ]

    resultado_bloco = pd.DataFrame(candidatos[colunas_resultado])
    resultados_intersecao.append(resultado_bloco)
    total_intersecoes_validas += len(resultado_bloco)

    print(
        f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
        f'IDs novos={len(representantes_novos):,} | '
        f'repetições puladas={len(bloco) - len(representantes_novos):,} | '
        f'candidatos={len(mascara_area_positiva):,} | '
        f'interseções com área={len(resultado_bloco):,}'
    )

    del (
        bloco, representantes, representantes_novos, gdf_ids, candidatos,
        geometria_gleba_wgs84, geometria_quilombola_wgs84,
        geometria_gleba_area, geometria_quilombola_area,
        geometria_intersecao_area, geometria_intersecao_area_valida,
        geometria_intersecao_wgs84, mascara_area_positiva, resultado_bloco,
    )
    gc.collect()

print('\nPROCESSAMENTO ESPACIAL CONCLUÍDO')
print('=' * 90)
print('Linhas lidas:', f'{total_linhas_lidas:,}')
print('Linhas com ID e geometria:', f'{total_linhas_com_id_e_geometria:,}')
print('ID_GLEBA2 distintos:', f'{len(assinatura_geometria_por_id):,}')
print('IDs submetidos ao teste:', f'{total_ids_novos:,}')
print('Repetições que não foram testadas novamente:', f'{total_repeticoes_puladas:,}')
print('IDs com geometria válida:', f'{total_ids_geometria_valida:,}')
print('Geometrias inválidas corrigidas:', f'{total_geometrias_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_geometrias_descartadas:,}')
print('Candidatos espaciais:', f'{total_candidatos_espaciais:,}')
print('Interseções recuperadas:', f'{total_intersecoes_recuperadas:,}')
print('Falhas topológicas:', f'{total_falhas_topologicas:,}')
print('Contatos sem área positiva:', f'{total_contatos_sem_area:,}')
print('Interseções válidas ID_GLEBA2 × área quilombola:', f'{total_intersecoes_validas:,}')


Bloco 001: lidas=20,000 | IDs novos=18,663 | repetições puladas=1,337 | candidatos=17 | interseções com área=17
Bloco 002: lidas=20,000 | IDs novos=17,690 | repetições puladas=2,310 | candidatos=21 | interseções com área=21
Bloco 003: lidas=20,000 | IDs novos=17,256 | repetições puladas=2,744 | candidatos=17 | interseções com área=17
Bloco 004: lidas=20,000 | IDs novos=17,061 | repetições puladas=2,939 | candidatos=13 | interseções com área=13
Bloco 005: lidas=20,000 | IDs novos=16,706 | repetições puladas=3,294 | candidatos=18 | interseções com área=18
Bloco 006: lidas=20,000 | IDs novos=14,364 | repetições puladas=5,636 | candidatos=24 | interseções com área=24
Bloco 007: lidas=20,000 | IDs novos=15,135 | repetições puladas=4,865 | candidatos=7 | interseções com área=7
Bloco 008: lidas=20,000 | IDs novos=15,230 | repetições puladas=4,770 | candidatos=5 | interseções com área=5
Bloco 009: lidas=20,000 | IDs novos=14,954 | repetições puladas=5,046 | candidatos=8 | interseções com área=

In [7]:
# ============================================================
# 7. CONSOLIDAR E SALVAR O RESULTADO ÚNICO POR ID_GLEBA2
# ============================================================

if resultados_intersecao:
    intersecoes_por_id = pd.concat(
        resultados_intersecao,
        ignore_index=True,
        sort=False,
    )
else:
    intersecoes_por_id = pd.DataFrame(
        columns=[COLUNA_ID_GLEBA2, 'QUIL_ID_INTERNO']
    )

del resultados_intersecao
gc.collect()

quantidade_antes = len(intersecoes_por_id)

if not intersecoes_por_id.empty:
    intersecoes_por_id = (
        intersecoes_por_id
        .sort_values('AREA_INTERSECAO_HA', ascending=False)
        .drop_duplicates(
            subset=[COLUNA_ID_GLEBA2, 'QUIL_ID_INTERNO'],
            keep='first',
        )
        .reset_index(drop=True)
    )

print(
    'Duplicações técnicas ID_GLEBA2 × área quilombola removidas:',
    f'{quantidade_antes - len(intersecoes_por_id):,}',
)

intersecoes_por_id.to_csv(
    ARQUIVO_INTERSECAO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

# O resumo inclui todos os IDs, inclusive os sem interseção.
todos_ids = pd.DataFrame({
    COLUNA_ID_GLEBA2: list(assinatura_geometria_por_id.keys())
})

if intersecoes_por_id.empty:
    resumo_por_id = todos_ids.copy()
    resumo_por_id['TEM_INTERSECAO_QUILOMBOLA'] = False
    resumo_por_id['QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS'] = 0
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = 0.0
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = 0.0
else:
    resumo_positivo = (
        intersecoes_por_id.groupby(COLUNA_ID_GLEBA2, dropna=False)
        .agg(
            QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS=(
                'QUIL_ID_INTERNO', 'nunique'
            ),
            AREA_INTERSECAO_TOTAL_HA=('AREA_INTERSECAO_HA', 'sum'),
            PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA=(
                'PERCENTUAL_GLEBA_INTERSECTADA', 'max'
            ),
        )
        .reset_index()
    )

    resumo_por_id = todos_ids.merge(
        resumo_positivo,
        on=COLUNA_ID_GLEBA2,
        how='left',
        validate='one_to_one',
    )

    resumo_por_id['TEM_INTERSECAO_QUILOMBOLA'] = (
        resumo_por_id['QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS']
        .notna()
    )
    resumo_por_id['QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS'] = (
        resumo_por_id['QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS']
        .fillna(0)
        .astype('int64')
    )
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = (
        resumo_por_id['AREA_INTERSECAO_TOTAL_HA'].fillna(0.0)
    )
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = (
        resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'].fillna(0.0)
    )

    resumo_por_id = resumo_por_id[
        [
            COLUNA_ID_GLEBA2,
            'TEM_INTERSECAO_QUILOMBOLA',
            'QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS',
            'AREA_INTERSECAO_TOTAL_HA',
            'PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA',
        ]
    ]

resumo_por_id.to_csv(
    ARQUIVO_RESUMO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

print('\nResultado único salvo em:')
print(ARQUIVO_INTERSECAO_POR_ID)
print('\nResumo de todos os IDs salvo em:')
print(ARQUIVO_RESUMO_POR_ID)
print('\nIDs com interseção:', f"{resumo_por_id['TEM_INTERSECAO_QUILOMBOLA'].sum():,}")
print('IDs sem interseção:', f"{(~resumo_por_id['TEM_INTERSECAO_QUILOMBOLA']).sum():,}")


Duplicações técnicas ID_GLEBA2 × área quilombola removidas: 0

Resultado único salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/quilombolas/resultados_por_id_gleba2/intersecao_quilombolas_unica_por_id_gleba2.csv

Resumo de todos os IDs salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/quilombolas/resultados_por_id_gleba2/resumo_quilombolas_por_id_gleba2.csv

IDs com interseção: 300
IDs sem interseção: 395,275


In [8]:
# ============================================================
# 8. DEVOLVER A RESPOSTA A TODAS AS LINHAS DO SICOR
# ============================================================

# A interseção não é recalculada aqui. O merge apenas replica a resposta
# de cada ID_GLEBA2 para todos os REF_BACEN, ordens, índices e anos ligados a ele.

colunas_espaciais_para_merge = [
    coluna
    for coluna in intersecoes_por_id.columns
    if coluna != COLUNA_ID_GLEBA2
]

escrever_cabecalho = True
total_linhas_originais_reprocessadas = 0
total_linhas_expandidas = 0

leitor_expandir = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_expandir, start=1):
    bloco = normalizar_colunas(bloco)
    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    total_linhas_originais_reprocessadas += len(bloco)

    if intersecoes_por_id.empty:
        expandido = pd.DataFrame()
    else:
        expandido = bloco.merge(
            intersecoes_por_id,
            on=COLUNA_ID_GLEBA2,
            how='inner',
            validate='many_to_many',
            suffixes=('', '_INTERSECAO'),
        )

    if expandido.empty:
        print(f'Bloco {numero_bloco:03d}: nenhuma linha com interseção')
        del bloco, expandido
        gc.collect()
        continue

    expandido.to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        mode='w' if escrever_cabecalho else 'a',
        header=escrever_cabecalho,
        index=False,
        sep=SEPARADOR_SAIDA,
        decimal=',',
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
    )

    total_linhas_expandidas += len(expandido)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: originais={len(bloco):,} | '
        f'linhas de interseção expandidas={len(expandido):,}'
    )

    del bloco, expandido
    gc.collect()

if escrever_cabecalho:
    # Nenhuma interseção: ainda cria um CSV vazio com cabeçalho conhecido.
    pd.DataFrame(
        columns=[*colunas_sicor, *colunas_espaciais_para_merge]
    ).to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        index=False,
        sep=SEPARADOR_SAIDA,
        encoding='utf-8-sig',
    )

if total_linhas_originais_reprocessadas != total_linhas_lidas:
    raise RuntimeError(
        'A quantidade de linhas relidas para a expansão difere da '
        'quantidade lida na etapa espacial.'
    )

print('\nRESULTADO DETALHADO EXPANDIDO')
print('=' * 90)
print('Linhas originais relidas:', f'{total_linhas_originais_reprocessadas:,}')
print('Linhas finais de interseção:', f'{total_linhas_expandidas:,}')
print('Arquivo salvo em:')
print(ARQUIVO_INTERSECAO_EXPANDIDA)


Bloco 001: originais=20,000 | linhas de interseção expandidas=17
Bloco 002: originais=20,000 | linhas de interseção expandidas=25
Bloco 003: originais=20,000 | linhas de interseção expandidas=21
Bloco 004: originais=20,000 | linhas de interseção expandidas=14
Bloco 005: originais=20,000 | linhas de interseção expandidas=20
Bloco 006: originais=20,000 | linhas de interseção expandidas=29
Bloco 007: originais=20,000 | linhas de interseção expandidas=8
Bloco 008: originais=20,000 | linhas de interseção expandidas=5
Bloco 009: originais=20,000 | linhas de interseção expandidas=9
Bloco 010: originais=20,000 | linhas de interseção expandidas=10
Bloco 011: originais=20,000 | linhas de interseção expandidas=30
Bloco 012: originais=20,000 | linhas de interseção expandidas=18
Bloco 013: originais=20,000 | linhas de interseção expandidas=7
Bloco 014: originais=20,000 | linhas de interseção expandidas=13
Bloco 015: originais=20,000 | linhas de interseção expandidas=19
Bloco 016: originais=20,000 |

In [9]:
# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print('ARQUIVOS GERADOS')
print('=' * 100)

for caminho in [
    ARQUIVO_INTERSECAO_POR_ID,
    ARQUIVO_INTERSECAO_EXPANDIDA,
    ARQUIVO_RESUMO_POR_ID,
]:
    print(f"{'OK' if caminho.exists() else 'NÃO CRIADO'} | {caminho}")

print('\nAMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2')
display(intersecoes_por_id.head(5))

print('\nAMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO')
display(resumo_por_id.head(5))

print('\nESTATÍSTICAS')
print('ID_GLEBA2 distintos analisados:', f'{len(resumo_por_id):,}')
print('ID_GLEBA2 com interseção:', f"{resumo_por_id['TEM_INTERSECAO_QUILOMBOLA'].sum():,}")
print('Pares ID_GLEBA2 × área quilombola:', f'{len(intersecoes_por_id):,}')

if not intersecoes_por_id.empty:
    print(
        'Área total das interseções únicas (ha):',
        f"{intersecoes_por_id['AREA_INTERSECAO_HA'].sum():,.4f}",
    )


ARQUIVOS GERADOS
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/quilombolas/resultados_por_id_gleba2/intersecao_quilombolas_unica_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/quilombolas/resultados_por_id_gleba2/intersecao_sicor_quilombolas_expandida_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/quilombolas/resultados_por_id_gleba2/resumo_quilombolas_por_id_gleba2.csv

AMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2


,ID_GLEBA2,QUIL_CD_QUILOMB,QUIL_CD_SR,QUIL_NR_PROCESS,QUIL_NM_COMUNID,QUIL_NM_MUNICIP,QUIL_CD_UF,QUIL_DT_PUBLICA,QUIL_DT_PUBLIC1,QUIL_NR_FAMILIA,QUIL_DT_TITULAC,QUIL_NR_AREA_HA,QUIL_NR_PERIMET,QUIL_CD_SIPRA,QUIL_OB_DESCRIC,QUIL_ST_TITULAD,QUIL_DT_DECRETO,QUIL_TP_LEVANTA,QUIL_NR_ESCALAO,QUIL_AREA_CALC,QUIL_PERIMETRO,QUIL_ESFERA,QUIL_FASE,QUIL_RESPONSAVE,QUIL_ID_INTERNO,QUIL_AREA_TOTAL_HA,AREA_GLEBA_HA_CALCULADA,AREA_INTERSECAO_HA,PERCENTUAL_GLEBA_INTERSECTADA,GEOMETRIA_QUILOMBOLA_WKT,GEOMETRIA_INTERSECAO_WKT,TIPO_IMPEDIMENTO
0,237359,NaN,SR-26,54400.000797/2004-01,BARRA DO AROEIRA,SANTA TEREZA DONOVO ACORDO LAGOA DO TOCANTINS,TO,24/11/2011,None,174.0,None,62315.3819,NaN,None,None,None,None,None,None,62519.3811,1.293801e+06,FEDERAL,RTID,INCRA,17,62890.090325,601.046477,601.046477,100.0,"POLYGON ((-47.773709160027856 -10.317889785144663, -47.77294876561689 -10.31672816573288, -47.77309855183342 -10.315465671844427, -47.773467073053986 -10.314719153139363, -47.7...","POLYGON ((-47.64871598404398 -10.127468020812893, -47.64631602730759 -10.120689041446793, -47.642726028178345 -10.122386016689301, -47.64047799167286 -10.128031977939859, -47.6...",Território Quilombola
1,218097,NaN,SR-26,54400.000797/2004-01,BARRA DO AROEIRA,SANTA TEREZA DONOVO ACORDO LAGOA DO TOCANTINS,TO,24/11/2011,None,174.0,None,62315.3819,NaN,None,None,None,None,None,None,62519.3811,1.293801e+06,FEDERAL,RTID,INCRA,17,62890.090325,354.559551,354.559551,100.0,"POLYGON ((-47.773709160027856 -10.317889785144663, -47.77294876561689 -10.31672816573288, -47.77309855183342 -10.315465671844427, -47.773467073053986 -10.314719153139363, -47.7...","POLYGON ((-47.6074124078528 -10.222788876527867, -47.60693391250715 -10.223300673983301, -47.60694031406616 -10.224077636831318, -47.60884497530049 -10.224082059850726, -47.609...",Território Quilombola
2,55626,NaN,SR-16,54290.004222/2006-14,FAMÍLIA JARCEM,RIO BRILHANTE,MS,16/08/2017,None,97.0,None,197.6181,23763.7,None,None,None,None,GPS,1:40.000,2166.0237,2.465930e+05,FEDERAL,RTID,INCRA,338,2166.286993,348.396581,348.396581,100.0,"MULTIPOLYGON (((-54.919400395556 -21.603717293056, -54.912311009444 -21.6090148525, -54.909769673889 -21.610912904167, -54.90979443 -21.611097226944, -54.91016186777801 -21.611...","POLYGON ((-54.96633895855168 -21.585432023143984, -54.96578996063447 -21.585820989279057, -54.965195033151865 -21.58658001742137, -54.96454200873261 -21.586208043342467, -54.96...",Território Quilombola
3,246256,NaN,SR-26,54400.000797/2004-01,BARRA DO AROEIRA,SANTA TEREZA DONOVO ACORDO LAGOA DO TOCANTINS,TO,24/11/2011,None,174.0,None,62315.3819,NaN,None,None,None,None,None,None,62519.3811,1.293801e+06,FEDERAL,RTID,INCRA,17,62890.090325,335.772358,335.772358,100.0,"POLYGON ((-47.773709160027856 -10.317889785144663, -47.77294876561689 -10.31672816573288, -47.77309855183342 -10.315465671844427, -47.773467073053986 -10.314719153139363, -47.7...","POLYGON ((-47.63770575795427 -10.144086152381153, -47.643713908063305 -10.144804349221845, -47.64414311621275 -10.143832666472415, -47.646160092218246 -10.143832734400057, -47....",Território Quilombola
4,290028,NaN,SR-26,54400.000797/2004-01,BARRA DO AROEIRA,SANTA TEREZA DONOVO ACORDO LAGOA DO TOCANTINS,TO,24/11/2011,None,174.0,None,62315.3819,NaN,None,None,None,None,None,None,62519.3811,1.293801e+06,FEDERAL,RTID,INCRA,17,62890.090325,329.688795,329.688795,100.0,"POLYGON ((-47.773709160027856 -10.317889785144663, -47.77294876561689 -10.31672816573288, -47.77309855183342 -10.315465671844427, -47.773467073053986 -10.314719153139363, -47.7...","POLYGON ((-47.61261918121192 -10.20598850008938, -47.610526579143404 -10.204512206218507, -47.613597978258774 -10.203327446787448, -47.61461067909093 -10.201648459804503, -47.6...",Território Quilombola



AMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO


,ID_GLEBA2,TEM_INTERSECAO_QUILOMBOLA,QUANTIDADE_AREAS_QUILOMBOLAS_INTERSECTADAS,AREA_INTERSECAO_TOTAL_HA,PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA
0,1,False,0,0.0,0.0
1,2,False,0,0.0,0.0
2,3,False,0,0.0,0.0
3,4,False,0,0.0,0.0
4,5,False,0,0.0,0.0



ESTATÍSTICAS
ID_GLEBA2 distintos analisados: 395,575
ID_GLEBA2 com interseção: 300
Pares ID_GLEBA2 × área quilombola: 303
Área total das interseções únicas (ha): 13,320.2952
